In [16]:
import torch
from dataclasses import dataclass
from T5CLIP import ConditioningEncoders
from model import mmDiT
from vae import VAE
# ---------------HYPER-PARAMS-------------------------------------
@dataclass
class TrainConfig:
    lr: float = 1e-4
    batch_size: int = 8
    total_steps: int = 15000 # 50k steps is ideal.
    lr_cooldown_steps: int = 20000
    starting_step: int = 0
    logging_interval: int = 100
    grad_accum_steps: int = 2 # not really needed
    save_dir: str = "./saved_models/"
    ckpt: str = "flowmatch-ckpt.pt"

@dataclass
class VAEConfig:
    kl_beta: float = 5e-2
    fft_weight: float = 0.05
    lpips_weight: float = 0.5
    latent_channels: tuple[int, ...] = (128, 256, 512, 512)
    z_channels: int = 32
    n_heads: int = 8
    resnet_blocks: int  = 2

@dataclass
class mmDiTConfig:
    d_model: int = 768
    n_heads: int = 32
    n_blocks: int = 16
    patch_size: int = 2
    color_channels: int = 32 # z_channels
    height: int = 32 # latent
    width: int = 32 # latent
    t_channels: int = 64
    p_cond: float = 0.2
    
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
print(device)

cuda


In [9]:
vae_cfg = VAEConfig()
dit_cfg = mmDiTConfig()
train_cfg = TrainConfig()

T5CLIP = ConditioningEncoders(device=device, debugging=False)

model = mmDiT(
    d_model=dit_cfg.d_model,
    n_heads=dit_cfg.n_heads,
    n_blocks=dit_cfg.n_blocks,
    patch_size=dit_cfg.patch_size,
    color_channels=dit_cfg.color_channels, 
    height=dit_cfg.height, 
    width=dit_cfg.width, 
    t_channels=dit_cfg.t_channels,
    pooled_dim=T5CLIP.pooled_dim,
    t5_dim=T5CLIP.t5_dim
).to(device)
vae = VAE(
    n_channels=3,
    latent_channels=vae_cfg.latent_channels,
    z_channels=vae_cfg.z_channels,
    n_heads=vae_cfg.n_heads,
    resnet_blocks=vae_cfg.resnet_blocks
).to(device).to(memory_format=torch.channels_last)
vae.eval()
for param in vae.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [3]:
vae.load_state_dict(torch.load(train_cfg.save_dir + "vae_compiled_10000_ema.pt", weights_only=True))
checkpoint = torch.load("./saved_models/flowmatch-ckpt10000.pt", weights_only=True)
# model.load_state_dict(checkpoint['ema'])
model.load_state_dict(checkpoint['model'])

<All keys matched successfully>

In [ ]:
model.eval()
from torchvision.utils import save_image
with torch.no_grad():
    t5, pooled, attn_mask = T5CLIP.encode(captions=["A cat resting in a grassy field.", "Hi"])
    total_timesteps = 1
    batch = 8
    output = model.sample(total_timesteps, t5, pooled, attn_mask)
    
    print(output[total_timesteps - 1].shape)
    final_image = vae.decode(output[total_timesteps - 1])
    save_image(final_image, "sampled_image.png")

RuntimeError: The size of tensor a (2) must match the size of tensor b (32) at non-singleton dimension 3

In [37]:
final_image = vae.decode(output[-1])
save_image(final_image, "sampled_image.png")